# 01 — Data Exploration
**Project:** ViT Reliability & Explainability Under Medical Distribution Shift
**Author:** Sosna Worku

**Goal:** Explore NIH ChestX-ray14 — class distribution, patient stats, sample images, bounding boxes.

---
Run all cells top to bottom every new Colab session.

## 0. Setup — Run every session

In [ ]:
import os, sys

REPO_PATH = '/content/vit-medical-shift'
GITHUB    = 'https://github.com/sossyh/vit-medical-shift.git'

if os.path.exists(REPO_PATH):
    os.system(f'git -C {REPO_PATH} pull origin main')
else:
    os.system(f'git clone {GITHUB} {REPO_PATH}')

sys.path.insert(0, REPO_PATH)
print('Repo ready!')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

In [ ]:
!pip install -q timm torchmetrics grad-cam einops pyyaml
print('Packages ready!')

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 1. Paths

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/data/nih'
CSV_PATH   = f'{DRIVE_ROOT}/Data_Entry_2017.csv'
BBOX_PATH  = f'{DRIVE_ROOT}/BBox_List_2017.csv'
IMG_DIR    = f'{DRIVE_ROOT}/images'

print('CSV exists  :', os.path.exists(CSV_PATH))
print('BBox exists :', os.path.exists(BBOX_PATH))
print('Images dir  :', os.path.exists(IMG_DIR))
if os.path.exists(IMG_DIR):
    print('Images found:', f'{len(os.listdir(IMG_DIR)):,}')

## 2. Load & Inspect CSV

In [ ]:
import pandas as pd
import numpy as np
from src.utils import NIH_LABELS

df = pd.read_csv(CSV_PATH)
print('Total images    :', len(df))
print('Unique patients :', df['Patient ID'].nunique())
print('\nGender:')
print(df['Patient Gender'].value_counts())
print('\nAge stats:')
print(df['Patient Age'].describe())
df.head()

## 3. Class Distribution

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

# Count each label
label_counts = {}
for label in NIH_LABELS:
    label_counts[label] = df['Finding Labels'].str.contains(label).sum()
label_counts['No Finding'] = df['Finding Labels'].str.contains('No Finding').sum()
label_counts = dict(sorted(label_counts.items(), key=lambda x: x[1], reverse=True))

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(list(label_counts.keys()), list(label_counts.values()),
               color='#378ADD', edgecolor='white')
ax.set_xlabel('Number of images')
ax.set_title('NIH ChestX-ray14 — Class Distribution', fontsize=13, fontweight='bold')
ax.invert_yaxis()
for bar, val in zip(bars, label_counts.values()):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=8)
plt.tight_layout()

os.makedirs(f'{REPO_PATH}/results/figures', exist_ok=True)
plt.savefig(f'{REPO_PATH}/results/figures/class_distribution.png', dpi=150)
plt.show()
print('Saved!')

## 4. Multi-label Distribution

In [ ]:
df['n_labels'] = df['Finding Labels'].apply(
    lambda x: 0 if x == 'No Finding' else len(x.split('|'))
)

fig, ax = plt.subplots(figsize=(8, 4))
df['n_labels'].value_counts().sort_index().plot(
    kind='bar', ax=ax, color='#378ADD', edgecolor='white'
)
ax.set_xlabel('Number of conditions per image')
ax.set_ylabel('Count')
ax.set_title('Multi-label Distribution', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{REPO_PATH}/results/figures/multilabel_dist.png', dpi=150)
plt.show()

## 5. Sample Images

In [ ]:
from PIL import Image
from src.dataset import NIHChestDataset, get_transforms

def show_samples(df, img_dir, label=None, n=8, cols=4):
    subset = (
        df[df['Finding Labels'].str.contains(label)].sample(n, random_state=42)
        if label else df.sample(n, random_state=42)
    )
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*3, rows*3))
    axes = axes.flatten()
    for i, (_, row) in enumerate(subset.iterrows()):
        img_path = os.path.join(img_dir, row['Image Index'])
        if os.path.exists(img_path):
            axes[i].imshow(Image.open(img_path).convert('RGB'), cmap='gray')
        axes[i].set_title(row['Finding Labels'][:25], fontsize=7)
        axes[i].axis('off')
    for j in range(i+1, len(axes)):
        axes[j].axis('off')
    fig.suptitle(label or 'Random samples', fontsize=12, fontweight='bold')
    plt.tight_layout()
    name = label.replace(' ', '_') if label else 'random'
    plt.savefig(f'{REPO_PATH}/results/figures/samples_{name}.png', dpi=150)
    plt.show()

show_samples(df, IMG_DIR)
show_samples(df, IMG_DIR, label='Pneumonia')
show_samples(df, IMG_DIR, label='Cardiomegaly')

## 6. Bounding Boxes (used for XAI evaluation in Week 3)

In [ ]:
bbox_df = pd.read_csv(BBOX_PATH)
print('BBox shape:', bbox_df.shape)
print('\nFindings with bounding boxes:')
print(bbox_df['Finding Label'].value_counts())
bbox_df.head()

In [ ]:
import matplotlib.patches as patches

def show_with_bbox(bbox_df, img_dir, finding='Cardiomegaly'):
    row = bbox_df[bbox_df['Finding Label'] == finding].iloc[0]
    img_path = os.path.join(img_dir, row['Image Index'])
    if not os.path.exists(img_path):
        print('Image not found.')
        return
    img = Image.open(img_path).convert('RGB')
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(img, cmap='gray')
    rect = patches.Rectangle(
        (row['Bbox [x'], row['y']),
        row['w'], row['h'],
        linewidth=2, edgecolor='red', facecolor='none'
    )
    ax.add_patch(rect)
    ax.set_title(f'{finding} — ground truth bounding box', fontsize=10)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(f'{REPO_PATH}/results/figures/bbox_{finding}.png', dpi=150)
    plt.show()

show_with_bbox(bbox_df, IMG_DIR, 'Cardiomegaly')
show_with_bbox(bbox_df, IMG_DIR, 'Pneumonia')

## 7. Test DataLoader from src/dataset.py

In [ ]:
from src.dataset import get_nih_loaders
from src.utils import get_device

device = get_device()

# Use subset=0.01 to test quickly (1% of data = ~1000 images)
train_loader, val_loader = get_nih_loaders(
    csv_path   = CSV_PATH,
    img_dir    = IMG_DIR,
    batch_size = 8,
    subset     = 0.01
)

print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')

# Test one batch
images, labels, names = next(iter(train_loader))
print(f'Image shape   : {images.shape}')
print(f'Label shape   : {labels.shape}')
print(f'Sample file   : {names[0]}')

## 8. Save Summary & Push to GitHub

In [ ]:
# Save class distribution CSV
summary = pd.DataFrame({
    'label'      : list(label_counts.keys()),
    'count'      : list(label_counts.values()),
    'prevalence' : [v/len(df) for v in label_counts.values()]
})
os.makedirs(f'{REPO_PATH}/results/metrics', exist_ok=True)
summary.to_csv(f'{REPO_PATH}/results/metrics/class_distribution.csv', index=False)
print('Saved class_distribution.csv!')

# Push results to GitHub
os.chdir(REPO_PATH)
!git config user.email 'sosworkuacha@gmail.com'
!git config user.name 'Sosna Worku'
!git add results/figures/ results/metrics/
!git commit -m 'week 1: data exploration figures and metrics'
!git push origin main